In [ ]:
# Storm Stats
import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
import rasterio
from rasterstats import zonal_stats


shapefile = "../../public/shapefiles/county.shp"

In [21]:
gdf = gpd.read_file(shapefile).dissolve(by="county", as_index=False).reset_index(drop=True)

print(gdf["county"].unique())  

for date in np.arange(11, 16):
  print(f"Processing date: 2026-03-{date:02d}")
  tif_path = f"./storm_data/2026_03_{date:02d}_cumulative.tif"

  for county in gdf["county"].unique():
      county_gdf = gdf[gdf["county"] == county]
      stats = zonal_stats(county_gdf, tif_path, stats=["mean", "min", "max"])

      # Round values
      rounded_stats = [
          {k: round(v, 2) if v is not None else None for k, v in s.items()}
          for s in stats
      ]

      print(f"Stats for {county}: {rounded_stats}")

['Hawaiʻi' 'Honolulu' 'Kauaʻi' 'Maui']
Processing date: 2026-03-11
Stats for Hawaiʻi: [{'min': 0.18, 'max': 36.55, 'mean': 7.99}]
Stats for Honolulu: [{'min': 26.38, 'max': 136.89, 'mean': 60.56}]
Stats for Kauaʻi: [{'min': 6.33, 'max': 213.48, 'mean': 76.88}]
Stats for Maui: [{'min': 5.88, 'max': 74.08, 'mean': 35.76}]
Processing date: 2026-03-12
Stats for Hawaiʻi: [{'min': 1.53, 'max': 47.31, 'mean': 17.13}]
Stats for Honolulu: [{'min': 37.31, 'max': 212.82, 'mean': 109.16}]
Stats for Kauaʻi: [{'min': 17.19, 'max': 356.41, 'mean': 131.41}]
Stats for Maui: [{'min': 16.14, 'max': 157.2, 'mean': 69.49}]
Processing date: 2026-03-13
Stats for Hawaiʻi: [{'min': 6.64, 'max': 271.92, 'mean': 44.93}]
Stats for Honolulu: [{'min': 96.28, 'max': 560.06, 'mean': 281.49}]
Stats for Kauaʻi: [{'min': 37.08, 'max': 524.43, 'mean': 236.57}]
Stats for Maui: [{'min': 96.9, 'max': 649.45, 'mean': 236.68}]
Processing date: 2026-03-14
Stats for Hawaiʻi: [{'min': 47.19, 'max': 730.32, 'mean': 253.52}]
Stats

In [ ]:
for m in range(1, 13):
  all_records = []

  for tif in sorted(glob.glob(os.path.join(tif_path, f"*{m:02d}.tif"))):
      parts = os.path.basename(tif).replace(".tif", "").split("_")
      year = int(parts[1])

      with rasterio.open(tif) as src:
          stats = zonal_stats(
              oahu,
              tif,
              stats=["mean"],
              nodata=src.nodata
          )

      mean_val = stats[0]["mean"]

      all_records.append({
          "year": year,
          "mean": mean_val
      })

  df = pd.DataFrame(all_records).sort_values("year")

  df["rank"] = df["mean"].rank(method="min", ascending=False)

  print(f"Month {m}: 2025 rank =", df.loc[df["year"] == 2025, "rank"].values[0])